# CAFA 6 - Baseline Models

Simple baselines to validate the pipeline:
1. **NaiveFrequency**: Predict training-set GO term frequency for every protein
2. Validate Fmax computation
3. Generate a sanity-check submission

In [ ]:
# === Google Colab Setup (skip if running locally) ===
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/AyushSharma173/CafaChallenge.git
    %cd CafaChallenge
    !pip install -q obonet biopython h5py lightgbm scikit-learn matplotlib seaborn pyyaml
    !pip install -q kaggle
    # Upload your kaggle.json: click the folder icon -> upload -> kaggle.json
    # Or run: from google.colab import files; files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    if os.path.exists('kaggle.json'):
        !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c cafa-6-protein-function-prediction -p data/raw
    !unzip -qo data/raw/cafa-6-protein-function-prediction.zip -d data/raw/
    %cd notebooks
    print('Colab setup complete!')

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
from pathlib import Path

from cafa6.data_loader import (
    load_fasta, load_train_terms, load_taxonomy,
    build_label_matrix, create_cv_split
)
from cafa6.go_utils import load_go_graph, ASPECT_FULL_NAME
from cafa6.metrics import compute_fmax
from cafa6.models import NaiveFrequency
from cafa6.submission import generate_submission, validate_submission

DATA_DIR = Path('../data/raw')

## 1. Load Data & Build Labels

In [ ]:
go_graph = load_go_graph(DATA_DIR / 'Train' / 'go-basic.obo')
terms_df = load_train_terms(DATA_DIR / 'Train' / 'train_terms.tsv')
taxonomy_df = load_taxonomy(DATA_DIR / 'Train' / 'train_taxonomy.tsv')

# Build label matrices for each ontology
label_data = {}
for aspect in ['P', 'F', 'C']:
    labels, pids, tids = build_label_matrix(
        terms_df, go_graph, aspect, min_count=50, propagate=True
    )
    label_data[aspect] = (labels, pids, tids)
    print(f'{ASPECT_FULL_NAME[aspect]}: {labels.shape[0]} proteins x {labels.shape[1]} terms, '
          f'density={labels.nnz/(labels.shape[0]*labels.shape[1]):.4f}')

## 2. Train/Val Split & Frequency Baseline

In [ ]:
results = {}

for aspect in ['P', 'F', 'C']:
    labels, pids, tids = label_data[aspect]
    
    # Split
    train_ids, val_ids = create_cv_split(pids, taxonomy_df, val_fraction=0.1, seed=42)
    train_mask = np.array([p in set(train_ids) for p in pids])
    val_mask = ~train_mask
    
    Y_train = labels[train_mask]
    Y_val = labels[val_mask]
    
    # Dummy X (frequency baseline ignores features)
    X_train = np.zeros((Y_train.shape[0], 1))
    X_val = np.zeros((Y_val.shape[0], 1))
    
    # Train
    model = NaiveFrequency()
    model.fit(X_train, Y_train)
    
    # Evaluate
    val_scores = model.predict(X_val)
    fmax, threshold = compute_fmax(Y_val, val_scores)
    results[aspect] = {'fmax': fmax, 'threshold': threshold}
    
    print(f'{ASPECT_FULL_NAME[aspect]}: Fmax={fmax:.4f} (t={threshold:.2f}), '
          f'train={Y_train.shape[0]}, val={Y_val.shape[0]}')

avg_fmax = np.mean([r['fmax'] for r in results.values()])
print(f'\nOverall average Fmax: {avg_fmax:.4f}')

## 3. Generate Sanity-Check Submission

Generate a submission for the test set using the frequency baseline.
This won't score well but validates the submission pipeline.

In [ ]:
# Load test proteins
test_seqs = load_fasta(DATA_DIR / 'Test' / 'testsuperset.fasta')
test_ids = list(test_seqs.keys())
print(f'Test proteins: {len(test_ids)}')

# Generate frequency-based predictions for all test proteins
predictions = {}
for aspect in ['P', 'F', 'C']:
    labels, pids, tids = label_data[aspect]
    
    model = NaiveFrequency()
    model.fit(np.zeros((labels.shape[0], 1)), labels)
    
    # All test proteins get the same frequency scores
    freqs = model.term_frequencies.flatten()
    
    for pid in test_ids:
        if pid not in predictions:
            predictions[pid] = {}
        for j, tid in enumerate(tids):
            if freqs[j] >= 0.01:  # min confidence
                predictions[pid][tid] = float(freqs[j])

print(f'Predictions for {len(predictions)} proteins')

# Generate submission
sub_df = generate_submission(
    predictions,
    go_graph=go_graph,
    output_path='../submissions/frequency_baseline.csv',
    propagate=True,
    min_confidence=0.01
)

# Validate
issues = validate_submission(
    '../submissions/frequency_baseline.csv',
    test_protein_ids=set(test_ids),
    valid_go_terms=set(go_graph.nodes())
)

In [ ]:
# Inspect submission
print(sub_df.head(20))
print(f'\nShape: {sub_df.shape}')
print(f'Confidence stats:\n{sub_df["confidence"].describe()}')